# 15.6 不确定性与校准 (Uncertainty & Calibration)

> 🕐 预估学习时间：35分钟

LLM 常常“自信地错”。不确定性估计与概率校准用于拒答路由、端云协同、评测可信度，是生产质量的关键一环。

本节涵盖：
- 置信度 vs 准确率
- Expected Calibration Error (ECE)
- Temperature Scaling
- 语义熵 / 自一致性不确定性
- 拒答与路由策略


## 1. 可靠性图与 ECE

将样本按置信度分箱，比较每箱平均置信度与实际准确率。ECE 是各箱差距的加权平均。


In [ ]:
import torch
import torch.nn.functional as F
import math

torch.manual_seed(0)

# Synthetic overconfident classifier logits
n, c = 1000, 5
logits = torch.randn(n, c) * 3.0
# plant labels correlated but noisy
probs = F.softmax(logits, dim=-1)
pred = probs.argmax(-1)
conf = probs.max(-1).values
# true labels: 60% correct
correct_mask = torch.rand(n) < 0.6
labels = pred.clone()
labels[~correct_mask] = (pred[~correct_mask] + 1) % c
acc = (pred == labels).float().mean().item()


def ece(conf, pred, labels, n_bins=10):
    bins = torch.linspace(0, 1, n_bins + 1)
    ece_val = 0.0
    rows = []
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.sum() == 0:
            rows.append((bins[i].item(), bins[i+1].item(), 0, 0, 0))
            continue
        acc_bin = (pred[m] == labels[m]).float().mean().item()
        conf_bin = conf[m].mean().item()
        ece_val += m.float().mean().item() * abs(acc_bin - conf_bin)
        rows.append((bins[i].item(), bins[i+1].item(), m.sum().item(), acc_bin, conf_bin))
    return ece_val, rows


ece0, rows = ece(conf, pred, labels)
print('=== Calibration Diagnostics ===')
print(f'accuracy={acc:.3f} mean_conf={conf.mean():.3f} ECE={ece0:.4f}')
print(f'{"bin":>12} {"n":>5} {"acc":>7} {"conf":>7}')
for lo, hi, n_bin, a, cf in rows:
    if n_bin:
        print(f'{lo:.1f}-{hi:.1f} {n_bin:>5} {a:>7.3f} {cf:>7.3f}')
print(f'\nKey: Overconfidence shows up as mean_conf >> accuracy and large ECE.')


## 2. Temperature Scaling

在验证集上学习标量温度 T，推理时用 `softmax(logits / T)`。简单、不改模型参数，常作后处理校准。


In [ ]:
def nll_with_T(logits, labels, T):
    return F.cross_entropy(logits / T, labels)


T = torch.tensor(1.5, requires_grad=True)
opt = torch.optim.LBFGS([T], lr=0.1, max_iter=50)

def closure():
    opt.zero_grad()
    loss = nll_with_T(logits, labels, T.clamp_min(1e-3))
    loss.backward()
    return loss

opt.step(closure)
T_star = float(T.detach().clamp_min(1e-3))
probs_cal = F.softmax(logits / T_star, dim=-1)
conf_cal = probs_cal.max(-1).values
pred_cal = probs_cal.argmax(-1)
ece1, _ = ece(conf_cal, pred_cal, labels)
print('=== Temperature Scaling ===')
print(f'T*={T_star:.3f}')
print(f'ECE before={ece0:.4f} after={ece1:.4f}')
print(f'mean_conf before={conf.mean():.3f} after={conf_cal.mean():.3f}')
print(f'Key: Temperature scaling often cuts ECE without changing argmax predictions much.')



## 3. 语义熵与自一致性不确定性

对开放生成，token 概率未必等于语义正确概率。可多次采样，按语义聚类后计算熵；或看自一致性投票分散度。


In [ ]:
def semantic_entropy(answer_groups):
    '''answer_groups: list of cluster sizes for one question.'''
    total = sum(answer_groups)
    probs = torch.tensor([g / total for g in answer_groups], dtype=torch.float)
    return float(-(probs * probs.clamp_min(1e-12).log()).sum())


def self_consistency_uncertainty(votes):
    # fraction not equal to majority
    from collections import Counter
    c = Counter(votes)
    maj = c.most_common(1)[0][1]
    return 1.0 - maj / len(votes)


print('=== Generative Uncertainty ===')
print('confident math:', semantic_entropy([8, 1, 1]), 'self_cons_u=', self_consistency_uncertainty(['42']*8+['41','40']))
print('ambiguous:', semantic_entropy([3, 3, 2, 2]), 'self_cons_u=', self_consistency_uncertainty(['A','B','A','C','B','D','A','B']))
print(f'\nKey: Semantic entropy captures answer diversity beyond token-level confidence.')


## 4. 拒答 / 端云路由

高不确定性 → 拒答、要求澄清、或路由到更大模型/人工。阈值应用验证集按成本-风险曲线选择。


In [ ]:
def route(confidences, thr_local=0.8, thr_abstain=0.5):
    actions = []
    for c in confidences:
        if c >= thr_local:
            actions.append('local')
        elif c >= thr_abstain:
            actions.append('escalate')
        else:
            actions.append('abstain')
    return actions


# Use calibrated confidences
actions = route(conf_cal[:20].tolist())
from collections import Counter
print('=== Routing on calibrated confidence ===')
print(Counter(actions))
# simulate risk: wrong local answers
local_idx = [i for i, a in enumerate(actions) if a == 'local']
if local_idx:
    local_err = (pred_cal[local_idx] != labels[local_idx]).float().mean().item()
    print(f'local error rate on first-20 slice: {local_err:.3f}')
print(f'\nKey: Calibration makes confidence thresholds operational for abstention and routing.')


## 课后思考题

1. 为什么 Temperature Scaling 不改变 argmax，却能改善 ECE？
2. 语义熵在“多种正确表述”任务上会高估不确定性吗？如何缓解？
3. 拒答阈值如何同时兼顾用户体验与安全风险？
4. 分类校准方法迁移到工具调用/结构化输出时要注意什么？

---
> 本节涵盖了15.6 不确定性与校准的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
